# H&M M4 기준 arm — 300 epoch (개발 seed 43)

M5 A·B를 **같은 epoch에서** M4 단독과 비교하기 위한 기준 arm입니다. 확증된 H&M M4는 100 epoch 기록만 있고, 그 모듈은 `epochs=100`을 하드 고정하고 설정해시에 넣기 때문에 **연장이 불가능**합니다 — 보호분할 확증이 올라가 있는 모듈이라 손대지 않고, 같은 손실을 **별도 폴더에서 300 epoch** 돌립니다.

- 그래프: **이진**(균등 엣지가중). 학습 전에 M1이 쓴 인접행렬과 **같은지 검사**하고 다르면 멈춥니다 — 그래야 `A − 기준`이 M3 그래프 단독 효과가 됩니다.
- 손실: 확증된 원형 M4 `1 + 0.5·q_C·상품금액백분위·고객가격대적합도`. 가중치는 A가 쓰는 것과 **같은 배열**이고, 평균 원시가중 1.21998·CV 0.15905 일치 검사를 통과해야 학습이 시작됩니다.
- 학습 루프: A·B와 **같은 `_train_curve`**. 기준과 후보가 다른 루프에서 나오면 차이에 루프 차이가 섞입니다.

**2단계로 돌립니다.** 먼저 100 epoch(약 12.5h) → 동결된 M4@100을 재현하는지 확인 → 통과하면 같은 셀을 300으로 이어서(약 25h). 재현이 안 되면 거기서 멈추고 25시간을 쓰지 않습니다.

**표기.** epoch 100 M5 판정은 그대로 남습니다. 300은 사전등록된 연장 확인이고, 순열 arm이 없어 300 지점을 CLV 배정 효과로 귀속하지 않습니다. test/holdout은 열지 않습니다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from google.colab import drive

ROOT = Path('/content/drive/MyDrive/논문/data')
REFS = [ROOT/'results_v3_hm_clv_m3_centered_value_graph_s43_v1',
        ROOT/'results_v3_hm_m4_k1_assignment_control_hm2y_development_screen_v1']
if not os.path.ismount('/content/drive'):
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습을 시작하지 않았습니다.') from exc
for ref in REFS:
    if not list(ref.glob('*.json')):
        raise RuntimeError(f'기준 결과가 없습니다: {ref} — 재학습하지 않고 중단합니다.')

SOURCE_COMMIT = '250f7a63ea3eedd1146602d15ae2931c3fbed78b'
REPO = Path('/content/clv-m4ref300-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)   # 이전 세션의 클론이 남아 있을 때
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
for name in ('lightgcn_clv_v3', 'clv_m5_m3_m4_split_nv_hm2y_screen',
             'clv_m5_m3_m4_split_nv_hm2y_m4_reference_300'):
    if name in sys.modules and Path(sys.modules[name].__file__).resolve().parent != REPO.resolve():
        raise RuntimeError(f'{name}이 다른 코드에서 로드됐습니다. 런타임을 다시 시작하세요.')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
import clv_m5_m3_m4_split_nv_hm2y_m4_reference_300 as ref
cfg = ref.configure()
print(json.dumps({'source_commit': SOURCE_COMMIT, 'model_id': ref.MODEL_ID,
                  'out_dir': cfg.out_dir, 'candidates': str(ref.candidate_out_dir()),
                  'judge_epoch': ref.JUDGE_EPOCH}, ensure_ascii=False, indent=2))

## 1단계 — 데이터 준비 → 이진 그래프·가중치 검사 → 100 epoch

`prepare`가 A의 가중치 감사값 일치와 균등가중 그래프 = M1 이진 인접행렬을 확인합니다. 둘 중 하나라도 어긋나면 학습하지 않고 멈춥니다. 약 12.5시간.

In [ ]:
prepared = ref.prepare(cfg)
payload = ref.train(cfg, prepared, stop_epoch=100)

## 2단계 — 동결된 M4@100 재현 확인 (학습 없음)

같은 손실·같은 데이터·같은 seed지만 **학습 루프가 다른 모듈**에서 나온 수치와 맞는지 봅니다. `reproduces_frozen_m4_at_100`이 `False`면 **여기서 멈추고** 결과를 보고하세요 — 그 경우 300 기준은 확증된 M4의 연장이 아니라 별개 실행이 되고, 25시간을 더 쓸 근거가 사라집니다.

In [ ]:
gap = ref.reproduction_gap(payload, prepared)
print(gap.to_string(index=False))
print('\n재현 여부:', gap.attrs['reproduces_frozen_m4_at_100'],
      '| 최대 상대오차:', f"{gap.rel_gap.max():.3e}",
      '| 허용:', ref.REPRODUCTION_RELATIVE_TOLERANCE)

## 3단계 — 300 epoch까지 이어서 (약 25시간)

위 재현이 통과했을 때만 실행합니다. 같은 체크포인트에서 101 epoch부터 이어집니다.

In [ ]:
payload = ref.train(cfg, prepared, stop_epoch=300)

## 4단계 — 300 epoch 비교와 원본 ZIP

A·B가 300까지 끝나 있으면 같은 epoch에서 함께 읽습니다(아직이면 기준 arm만 나옵니다). `economic_at10_above_m4`가 "M4를 넘었는지", `economic_at10_noninferior_99pct_vs_m4`가 "99% 비열등인지"로 분리돼 있습니다 — 둘을 섞어 서술하지 마세요.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
result = ref.report(cfg, prepared, payload)
key = ['recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
       'price_purchase_amount_weighted_hit@10', 'vndcg@10',
       'price_purchase_amount_weighted_hit@20', 'vndcg@20',
       'price_purchase_amount_weighted_hit@50', 'vndcg@50']
view = result['comparison']
view = view[view.metric.isin(key) & view.epoch.eq(ref.JUDGE_EPOCH)]
for (model, reference), part in view.groupby(['model_id', 'reference']):
    print(f'\n=== epoch {ref.JUDGE_EPOCH} | {model} vs {reference} ===')
    print(part[['metric', 'reference_value', 'candidate_value', 'ratio']].to_string(index=False))
zip_path = Path('/content/clv_m5_m3_m4_split_nv_hm2y_m4_reference_300_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))